<a href="https://colab.research.google.com/github/Mattalukkal/Ai_ML/blob/main/Assignment_on_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
filepath = '/content/drive/MyDrive/Colab Notebooks/DATA/Mall_Customers.csv'
df = pd.read_csv(filepath)
df.head() # first 5 columns

Replace '?' with NaN

In [ ]:
df.replace('?', np.nan, inplace=True)


Handle Missing Values

In [ ]:
# Separate numerical & categorical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numerical with median
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical with mode
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)


Remove Income Column

In [ ]:
income = df['income']   # Save for later comparison
df.drop('income', axis=1, inplace=True)


4. Outlier Treatment (IQR Method)

In [ ]:
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]


In [ ]:
5. Encode Categorical Variables

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()


6. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)


7. Exploratory Data Analysis

Distribution Plot

In [ ]:
df_encoded.hist(figsize=(15,10))
plt.tight_layout()
plt.show()


Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df_encoded.corr(), cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()


8. KMeans Clustering

Elbow Method

In [ ]:
wcss = []

for i in range(2, 10):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(2, 10), wcss, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()


Choose Optimal K

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X_scaled)

print("Silhouette Score:", silhouette_score(X_scaled, labels))


9. Agglomerative Clustering

In [ ]:
agg = AgglomerativeClustering(n_clusters=3)
labels_agg = agg.fit_predict(X_scaled)

print("Agglomerative Silhouette Score:", silhouette_score(X_scaled, labels_agg))


10. PCA for 2D Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("KMeans Clusters (PCA Projection)")
plt.show()


11. Cluster Profiling

In [ ]:
 df_encoded['Cluster'] = labels
df_profile = df_encoded.groupby('Cluster').mean()

df_profile


12. Compare With Actual Income

In [ ]:
df_original = df.copy()
df_original['Cluster'] = labels
df_original['income'] = income

pd.crosstab(df_original['Cluster'], df_original['income'])


Policy suggestions or business implications

In [ ]:
Cluster 0:
- Higher education
- Higher capital gain
- Long working hours
→ Likely high-income professionals

Cluster 1:
- Medium education
- Moderate working hours
→ Middle-class workers

Cluster 2:
- Lower education
- Low capital gain
→ Vulnerable / low-income group
